# Expedia CORE build

This notebook is the reproducible entrypoint for `RAW → STAGING → CORE`. It preserves immutable RAW, materializes derived Parquet, runs the fixed dependency and distance checks, and writes the CORE schema/report/manifest.

The implementation intentionally stops at CORE: no CLEAN/SILVER layer, MARTS, dashboard, or sessionization is created. `search_params_id` is a parameter-combination surrogate, not a search or session identifier.

## Execution contract

- RAW views are read-only source inputs.
- STAGING keeps source grain and adds normalized dates, duplicate metadata, and quality flags.
- CORE performs deterministic exact deduplication, builds the fixed dimensions/facts, and validates median distance backoff.
- Only `data/derived/`, `docs/core_schema.md`, `docs/distance_imputation_report.md`, and `artifacts/core_manifest.json` are generated.

In [1]:
from pathlib import Path
import runpy

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
runpy.run_path(str(ROOT / 'tools' / 'build_core.py'), run_name='__main__')

{
  "build_timestamp": "2026-08-09T14:15:57+00:00",
  "staging_rows": 40198536,
  "core_event_rows": 40197567,
  "duplicates_removed": 969,
  "user_location_dimension": true,
  "distance_min_support": 5,
  "distance_final_missing": 9132755,
  "pk_checks_pass": true,
  "fk_checks_pass": true,
  "fanout_check_pass": true
}


{'__name__': '__main__',
 '__doc__': 'Materialize the Expedia STAGING and CORE layers.\n\nThe script is intentionally SQL-first.  It reads only the immutable raw views,\nmaterializes derived Parquet, and registers derived views in analytics.duckdb.\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/home/neukluzhiy/Desktop/Projects/HotelsBooking/tools/build_core.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': Mo

## Expected handoff

The generated `artifacts/core_manifest.json` records table grains, keys, row counts, source dependencies, and validation status. The two generated Markdown files document the physical CORE schema and distance holdout results.